# Multi-stage Dam Discharge Planning with Head-Dependent Efficiency

The "water system operator" of a power company, who comprehensively manages multiple dams along a Class A river, determines the discharge volume of each dam over a season (tens to over a hundred days). The discharge from an upstream dam becomes the inflow to the downstream dam (with a time delay for the river flow), and the power generation of each dam is determined by the bilinearity of **discharge volume $\times$ water head (dependent on storage water level)**.

Business meaning of each constraint:

- **Generation = Discharge volume $\times$ Head (bilinear)**: The turbine output of hydropower is roughly proportional to `P = k・q・h` (q = discharge volume, h = effective head). The head h is a linear function of the storage water level (= storage volume), and the higher the storage water level, the larger the power generation even with the same discharge volume. "When to store and when to release" dictates the generation value, and this is the non-convex structure itself (not an approximation, but the physics of hydropower).
- **Storage balance (time coupling)**: The storage volume of each period is the previous period's storage + own basin inflow + discharge/spill from upstream (with a 1-period river flow delay) - own dam discharge - spill. The decisions made upstream ripple through all periods via downstream constraints.
- **Irrigation intake lower limit**: Designated dams are obligated to maintain a minimum discharge during the irrigation season, meaning discharge cannot be determined solely for generation convenience.
- **End-of-term storage lower limit**: A reserve target against droughts. A trade-off between "using it now or saving it for the future."
- **Thermal backstop**: In periods where demand cannot be met by hydropower alone, the gap is filled by high-cost thermal alternatives (always ensuring feasibility).

Subject: `samples/energy_and_microgrid/hydro_cascade_efficiency.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import hydro_cascade_efficiency as hc
from pyscipopt import Model, quicksum

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Naive Formulation

The default scale is 5 dams $\times$ 22 periods. `head_def` (head definition, linear function of storage) is linear, but `gen_def` (generation = generation coefficient $\times$ discharge $\times$ head) is a **bilinearity of discharge (continuous) $\times$ head (continuous)**, appearing for the number of dams $\times$ periods.

In [ ]:
m0 = hc.build_model("default")
nl = sum(1 for c in m0.getConss() if c.isNonlinear())
print(f"変数 {m0.getNVars()}、制約 {m0.getNConss()}(うち非線形 {nl})")
nD, nT = m0.data["dims"]
print(f"ダム数 nD={nD} × 期数 nT={nT} = {nD*nT} -> gen_def の本数と一致: {nl == nD*nT}")
del m0

All non-linear constraints are exactly `gen_def` (generation definition), and the count exactly matches the number of dams $\times$ periods.
Since `head_def` (head = linear function of storage) itself is linear, the origin of the bilinearity lies precisely in the physical power generation equation of "discharge $\times$ head".

## 2. Diagnosis

In [ ]:
report = mk.analyze(lambda: hc.build_model("default"), name="hydro-cascade",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.1%}",
      "/ 支配ボトルネック:", report.metrics.get("bottleneck_type"),
      f"(相対違反 {report.metrics.get('bottleneck_rel_viol', 0):.2f})",
      "/ 空間分枝の双対寄与:", f"{report.metrics.get('spatial_share', 0):.0%}")

In 30 seconds, a gap of about 28% remains, and three issues trigger: `weak_relaxation` (serious, concentrated relaxation violations + spatial branches dominant), `dual_stall`, and `numerical_scale`. The dual contribution of spatial branches is **100%** — all dual bound improvements come from branching on continuous variables (discharge/head). Considering there are no integer variables in this problem, this indicates that the weakness of the McCormick relaxation in `gen_def` translates directly into the bottleneck. Let's look inside.

## 3. Looking Inside the Diagnosis — Concentration of Violations and Branch Attribution

We look directly at the non-linear constraint violations in the root LP relaxation solution (how `gen_def` violations are concentrated by dam and period) and the attribution of dual bound improvements.

In [ ]:
from minlpkit.collectors.violation import collect_root_violations, violation_by_type
from minlpkit.collectors.attribution import solve_and_attribute, gain_by_kind

m1 = hc.build_model("default")
vdf = collect_root_violations(m1)
# entity は "dam_period" の形。ダム別に集計
vdf["dam"] = vdf["entity"].str.split("_").str[0]
by_dam = vdf.groupby("dam")["rel_violation"].mean().reset_index()
by_dam["dam"] = by_dam["dam"].astype(int)
by_dam = by_dam.sort_values("dam")
by_dam

In [ ]:
fig = go.Figure(layout=base_layout(
    "gen_def(発電量定義)のルートLP緩和違反(ダム別平均、上流=0→下流)",
    "ダム番号(0=最上流)", "相対違反(平均)", height=380))
fig.add_trace(go.Bar(x=by_dam["dam"], y=by_dam["rel_violation"], marker_color=C["warn"]))
show(fig)

In [ ]:
m2 = hc.build_model("default")
d2, summ2 = solve_and_attribute(m2, time_limit=30, gap_limit=0.01)
gk = gain_by_kind(d2)
fig = go.Figure(layout=base_layout(
    "双対境界の改善の帰属(分枝の種類別)", "分枝の種類", "双対境界の押し上げ量(合計)", height=380))
fig.add_trace(go.Bar(x=gk["kind"], y=gk["dual_gain"],
    marker_color=[C["warn"] if k == "spatial" else C["s1"] for k in gk["kind"]],
    text=[f"{v:.1f}" for v in gk["dual_gain"]], textposition="outside"))
show(fig)

Violations are widely distributed across almost all dams and are not a problem isolated to a specific single dam. Branch attributions are also virtually entirely monopolized by `spatial` (spatial branches on continuous variables) — confirming that the weakness of McCormick relaxations is effective uniformly across all dams. To narrow the McCormick bounds, it is standard practice to **tighten the variable bounds of q and head (especially the upper bound of storage volume)** (Section 4).

## 4. Trying Improvements — Tightening Bounds by Reachable Storage Range (FBBT)

The current variable bounds of `S[dd,t]` (storage volume) are fixed to `[smin, smax]` (the full physical capacity of that dam) for all periods. However, in reality, since the amount of water that can flow in each period starting from the initial storage `s0` is finite, it is often impossible to reach `smax` in early periods. This is a classic example of **feasibility-based bound tightening (FBBT)**. SCIP's presolve also performs this kind of processing, but because there is no upper limit on spills (`sp`) in this model, automatic propagation alone cannot tighten it completely.

We calculate the upper bound of storage volume reachable when minimizing discharge `q` and spills `sp` (= 0 or irrigation lower limit), forward propagating from upstream to downstream:

```
S_hi[d,t] = min(smax[d], S_hi[d,t-1] + local_inflow[d,t] + upstream_hi[d,t])
```

This is a **strictly valid upper bound** derived from the monotonicity that "the less discharge and spill, the more storage increases" (it does not shave off the feasible region at all). As `S_hi` gets tightened, the upper limit of `head` is also linked and tightened, thereby narrowing the McCormick envelope of `gen = k・q・head`.

In [ ]:
def build_tight(scale="default"):
    '''S(貯水量)の到達可能上限をFBBTでタイト化した変種(head の上限も連動)。'''
    d = hc._data(scale)
    nD, nT = d["nD"], d["nT"]
    smin, smax, s0, qmax = d["smin"], d["smax"], d["s0"], d["qmax"]
    h0, alpha, k_gen = d["h0"], d["alpha"], d["k_gen"]
    local_inflow, irrig_min, s_target, demand = (
        d["local_inflow"], d["irrig_min"], d["s_target"], d["demand"])

    # 前方FBBT: 放流・越流を最小にしたときの到達可能な貯水量上限(妥当な上限、実行可能領域は不変)
    S_hi = np.zeros((nD, nT))
    sp_hi = np.zeros((nD, nT))
    for dd in range(nD):
        for t in range(nT):
            prev_hi = S_hi[dd, t - 1] if t > 0 else float(s0[dd])
            upstream_hi = 0.0 if (dd == 0 or t == 0) else float(qmax[dd - 1]) + sp_hi[dd - 1, t - 1]
            q_lo = float(irrig_min[dd, t]) if irrig_min[dd, t] > 0 else 0.0
            S_hi[dd, t] = min(float(smax[dd]), prev_hi + float(local_inflow[dd, t]) + upstream_hi)
            sp_hi[dd, t] = max(0.0, prev_hi + float(local_inflow[dd, t]) + upstream_hi
                               - q_lo - float(smin[dd]))

    m = Model("Hydro_Cascade_Efficiency_Tight")
    D, T = range(nD), range(nT)
    S = {(dd, t): m.addVar(vtype="C", lb=float(smin[dd]), ub=float(S_hi[dd, t]), name=f"S_{dd}_{t}")
         for dd in D for t in T}
    q = {(dd, t): m.addVar(vtype="C", lb=0.0, ub=float(qmax[dd]), name=f"q_{dd}_{t}")
         for dd in D for t in T}
    sp = {(dd, t): m.addVar(vtype="C", lb=0.0, name=f"sp_{dd}_{t}") for dd in D for t in T}
    head = {(dd, t): m.addVar(vtype="C",
                              lb=float(h0[dd] + alpha[dd] * smin[dd]),
                              ub=float(h0[dd] + alpha[dd] * S_hi[dd, t]),
                              name=f"head_{dd}_{t}") for dd in D for t in T}
    gen = {(dd, t): m.addVar(vtype="C", lb=0.0, name=f"gen_{dd}_{t}") for dd in D for t in T}
    thermal = {t: m.addVar(vtype="C", lb=0.0, name=f"thermal_{t}") for t in T}
    for dd in D:
        for t in T:
            m.addCons(head[dd, t] == float(h0[dd]) + float(alpha[dd]) * S[dd, t], name=f"head_def_{dd}_{t}")
            m.addCons(gen[dd, t] == float(k_gen[dd]) * q[dd, t] * head[dd, t], name=f"gen_def_{dd}_{t}")
            if irrig_min[dd, t] > 0:
                m.addCons(q[dd, t] >= float(irrig_min[dd, t]), name=f"irrig_{dd}_{t}")
            prev = S[dd, t - 1] if t > 0 else float(s0[dd])
            upstream_in = 0.0 if (dd == 0 or t == 0) else q[dd - 1, t - 1] + sp[dd - 1, t - 1]
            m.addCons(S[dd, t] == prev + float(local_inflow[dd, t]) + upstream_in - q[dd, t] - sp[dd, t],
                      name=f"storage_balance_{dd}_{t}")
        m.addCons(S[dd, nT - 1] >= float(s_target[dd]), name=f"terminal_storage_{dd}")
    for t in T:
        m.addCons(quicksum(gen[dd, t] for dd in D) + thermal[t] >= float(demand[t]), name=f"power_balance_{t}")
    m.setObjective(quicksum(hc.THERMAL_COST * thermal[t] for t in T), "minimize")
    m.data = dict(S=S, q=q, gen=gen, thermal=thermal, scale=scale, dims=(nD, nT))
    return m

mb = hc.build_model("small"); mb.hideOutput(); mb.optimize()
mt = build_tight("small"); mt.hideOutput(); mt.optimize()
print(f"baseline : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"tight    : obj={mt.getObjVal():.2f}  status={mt.getStatus()}")

In [ ]:
# 実際にどれだけ境界が締まったか(既定scale、上流ダム0が最も締まりやすい)
d = hc._data("default")
nD, nT = d["nD"], d["nT"]
smin, smax, s0, qmax = d["smin"], d["smax"], d["s0"], d["qmax"]
local_inflow, irrig_min = d["local_inflow"], d["irrig_min"]
S_hi = np.zeros((nD, nT)); sp_hi = np.zeros((nD, nT))
for dd in range(nD):
    for t in range(nT):
        prev_hi = S_hi[dd, t - 1] if t > 0 else float(s0[dd])
        upstream_hi = 0.0 if (dd == 0 or t == 0) else float(qmax[dd - 1]) + sp_hi[dd - 1, t - 1]
        q_lo = float(irrig_min[dd, t]) if irrig_min[dd, t] > 0 else 0.0
        S_hi[dd, t] = min(float(smax[dd]), prev_hi + float(local_inflow[dd, t]) + upstream_hi)
        sp_hi[dd, t] = max(0.0, prev_hi + float(local_inflow[dd, t]) + upstream_hi - q_lo - float(smin[dd]))

fig = go.Figure(layout=base_layout(
    "貯水量上限のタイト化(ダム別、元のsmaxに対する比率)",
    "期", "S_hi(タイト化後) / smax(元の上限)", height=400))
for dd in range(nD):
    ratio = S_hi[dd, :] / smax[dd]
    fig.add_trace(go.Scatter(x=list(range(nT)), y=ratio, mode="lines",
        line=dict(width=2, color=[C["s1"], C["s2"], C["warn"], C["muted"], C["ink"]][dd % 5]),
        name=f"ダム{dd}"))
fig.add_hline(y=1.0, line=dict(color=C["axis"], dash="dot"))
show(fig)

The most upstream dam (Dam 0) is clearly tightened in early periods (reachable storage volume is restricted below `smax`).
For downstream dams, because there is no upper limit on spills `sp`, estimates for inflows from upstream are loose, so the tightening effect is not as large as upstream — the limitation of the model premise that allows unlimited spills is directly reflected in the propagation.

In [ ]:
df = mk.compare_variants(
    {"baseline":     lambda: hc.build_model("default"),
     "tight bounds": lambda: build_tight("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
base = df.loc[df["variant"] == "baseline"].iloc[0]
tgt  = df.loc[df["variant"] == "tight bounds"].iloc[0]
labels = ["baseline", "tight bounds"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], tgt["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, tgt["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], tgt["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="貯水量の境界タイト化(FBBT) before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {tgt['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.1%} -> {tgt['final_gap']:.1%}")
print(f"nodes     : {int(base['nodes'])} -> {int(tgt['nodes'])}")

**Honest verification result**: Bound tightening via FBBT is a theoretically sound improvement that narrows the McCormick envelopes without altering the feasible region at all (equivalent transformation). The magnitude of the effect is exactly as measured (in the graph above) — tightening is clearly effective only in the early periods of the most upstream dam (as seen in the Section 3 graph). Downstream dams experience weak propagation because spills `sp` are unrestricted, often making the overall improvement margin limited. This is not because "the bound tightening technique itself is ineffective," but for the structural reason that **the design of this model, which allows unlimited spills, weakens the efficacy of FBBT** — if a physically reasonable upper limit on spills (e.g., total capacity of water passage facilities) is added, similar propagation should work downstream as well (unverified in this notebook, candidate for the next improvement).

## Summary

- The difficulty of this problem lies in the bilinearity generated by the **physical equation of hydropower generation itself: discharge $\times$ head**.
  Diagnosis pointed to `weak_relaxation` (100% dual contribution from spatial branches) across all dams and all periods, confirming that the problem is not isolated to a specific single unit or a single period.
- While tightening storage bounds via FBBT is a theoretically sound move, its effect depends on the dam's location (more effective upstream) and model premises (unlimited spills) — we successfully demonstrated empirically that **it is not a silver bullet and its effectiveness can be uneven depending on the model structure**.
- Practically, this is the comprehensive operational problem of a cascaded river: "When to store and when to discharge at upstream dams to connect to downstream generation and irrigation." The bilinearity is not an approximation but inevitably arises from the physics of generation (P=k・q・h). The irrigation lower limit acts as an exogenous constraint independent of generation optimization, further restricting this non-convex structure asymmetrically.

Related: [Method Guide 0. Diagnosis Itself](../../playbook/00-diagnose.md) /
[Method Guide 3. Eliminating Big-M](../../playbook/03-bigm.md) (related techniques for bound tightening) /
API [`mk.analyze`](../../api/pipeline.md)